# Model Training & Evaluation
## Portal Pelaporan Tindak Kriminal Terpadu



## 1. Import & Konfigurasi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
import warnings
import os
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
matplotlib.use('Agg')

# ── Path config ───────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH  = ROOT / "data" / "dataset_siap_training.csv"
MODEL_DIR  = ROOT / "backend" / "ml" / "models"
DOCS_DIR   = ROOT / "docs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT      : {ROOT}")
print(f"DATA_PATH : {DATA_PATH}")
print(f"Exists    : {DATA_PATH.exists()}")


ROOT      : C:\Users\thori\Documents\Semester 4\uas\crime-reporting-structure\crime-reporting
DATA_PATH : C:\Users\thori\Documents\Semester 4\uas\crime-reporting-structure\crime-reporting\data\dataset_siap_training.csv
Exists    : True


## 2. Load & Split Data

In [13]:
df = pd.read_csv(DATA_PATH)
print(f"Total dataset : {len(df):,} rows")

# ── Gunakan DATA REAL ONLY untuk training & evaluasi ─────────
# Data sintetis (SMOTE) sudah ada di CSV — kita pisahkan
# Test set WAJIB dari data real agar metrik representatif
df_real      = df[df['is_synthetic'] == 'No'].copy()
df_synthetic = df[df['is_synthetic'] == 'Yes'].copy()

print(f"Data real     : {len(df_real):,} rows")
print(f"Data sintetis : {len(df_synthetic):,} rows")
print()
print("Distribusi label (data real):")
print(df_real['label_urgensi'].value_counts().to_string())


Total dataset : 3,771 rows
Data real     : 2,413 rows
Data sintetis : 1,358 rows

Distribusi label (data real):
label_urgensi
Tinggi    1257
Sedang     986
Rendah     170


In [14]:
# ── Train/Test Split (stratified) ────────────────────────────
X_real = df_real['deskripsi_bersih'].fillna('').values
y_real = df_real['label_urgensi'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_real, y_real,
    test_size=0.2,
    random_state=42,
    stratify=y_real
)

# Gabungkan data sintetis ke TRAIN SET SAJA
X_train_raw = np.concatenate([X_train_raw, df_synthetic['deskripsi_bersih'].fillna('').values])
y_train      = np.concatenate([y_train, df_synthetic['label_urgensi'].values])

print(f"Train set : {len(X_train_raw):,} rows (real + sintetis)")
print(f"Test set  : {len(X_test_raw):,}  rows (real only)")
print()
print("Distribusi y_train:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c}")
print()
print("Distribusi y_test (data real):")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c}")


Train set : 3,288 rows (real + sintetis)
Test set  : 483  rows (real only)

Distribusi y_train:
  Rendah: 1223
  Sedang: 1060
  Tinggi: 1005

Distribusi y_test (data real):
  Rendah: 34
  Sedang: 197
  Tinggi: 252


In [15]:
# ── TF-IDF Vectorizer ─────────────────────────────────────────
# Fit pada train set SAJA — tidak boleh lihat test set
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,     # log(tf) — mengurangi dominasi kata sangat sering
    strip_accents='unicode',
)

X_train = vectorizer.fit_transform(X_train_raw)
X_test  = vectorizer.transform(X_test_raw)

print(f"Vocabulary size : {len(vectorizer.vocabulary_):,}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_test shape    : {X_test.shape}")

# Simpan vectorizer
joblib.dump(vectorizer, MODEL_DIR / "vectorizer.pkl")
print(f"\n✅ Vectorizer disimpan: {MODEL_DIR / 'vectorizer.pkl'}")


Vocabulary size : 10,000
X_train shape   : (3288, 10000)
X_test shape    : (483, 10000)

✅ Vectorizer disimpan: C:\Users\thori\Documents\Semester 4\uas\crime-reporting-structure\crime-reporting\backend\ml\models\vectorizer.pkl


## 3. PyCaret — Perbandingan Model

PyCaret digunakan untuk membandingkan 15+ algoritma secara otomatis
dan memberikan **justifikasi ilmiah** dalam memilih 3 model baseline.

> Bukan memilih model secara subjektif, melainkan berbasis data.


In [16]:
from pycaret.classification import setup, compare_models, pull

# PyCaret bekerja dengan DataFrame — gunakan data train
df_pycaret = pd.DataFrame({
    'teks'   : X_train_raw,
    'label'  : y_train
})

print("Setup PyCaret...")
clf_setup = setup(
    data            = df_pycaret,
    target          = 'label',
    train_size      = 0.8,
    fold            = 5,
    session_id      = 42,
    text_features   = ['teks'],
    verbose         = False,
)
print("✅ Setup selesai")


RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=10, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')

In [ ]:
print("Membandingkan semua model (estimasi 5-10 menit)...")
best_models = compare_models(
    n_select     = 3,       # ambil top 3
    sort         = 'F1',    # sort berdasarkan F1 (lebih fair untuk multiclass)
    verbose      = True,
    exclude      = ['catboost'],  # exclude karena butuh install terpisah
)

# Tampilkan hasil perbandingan
results_df = pull()
print("\n=== HASIL PERBANDINGAN MODEL ===")
print(results_df[['Model', 'Accuracy', 'F1', 'Prec.', 'Recall', 'TT (Sec)']].to_string())

# Identifikasi top 3
print("\n✅ Top 3 model yang dipilih untuk baseline:")
for i, m in enumerate(best_models):
    print(f"  {i+1}. {type(m).__name__}")


In [ ]:
# ── Visualisasi hasil PyCaret ─────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

top_n = min(10, len(results_df))
models_viz  = results_df['Model'].head(top_n).values
f1_scores   = results_df['F1'].head(top_n).values
acc_scores  = results_df['Accuracy'].head(top_n).values

x = np.arange(top_n)
w = 0.35
bars1 = ax.bar(x - w/2, f1_scores,   w, label='F1 Score',  color='#3498db', alpha=0.85)
bars2 = ax.bar(x + w/2, acc_scores,  w, label='Accuracy',  color='#e74c3c', alpha=0.85)

ax.set_title('Perbandingan Model — PyCaret AutoML
(Top 10 dari seluruh algoritma)', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models_viz, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(DOCS_DIR / '08_pycaret_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan ke docs/08_pycaret_comparison.png")


## 4. Training Baseline Model + MLflow Tracking

Training ulang top 3 model dari PyCaret dengan MLflow tracking penuh.
Setiap model dicatat sebagai satu **run** di MLflow.


In [ ]:
# ── Definisi 3 baseline model ─────────────────────────────────
# Model ini dipilih berdasarkan hasil PyCaret di atas
BASELINE_MODELS = {
    "LogisticRegression": LogisticRegression(
        C=1.0, max_iter=1000, random_state=42,
        multi_class='multinomial', solver='lbfgs'
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "LinearSVC": LinearSVC(
        C=1.0, max_iter=2000, random_state=42
    ),
}

def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    """Train dan evaluasi satu model, return dict metrics"""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    return {
        "accuracy"         : accuracy_score(y_te, y_pred),
        "f1_macro"         : f1_score(y_te, y_pred, average='macro'),
        "f1_weighted"      : f1_score(y_te, y_pred, average='weighted'),
        "precision_macro"  : precision_score(y_te, y_pred, average='macro'),
        "recall_macro"     : recall_score(y_te, y_pred, average='macro'),
    }, y_pred

print("✅ Model definisi siap")
print("Model yang akan ditraining:")
for name in BASELINE_MODELS:
    print(f"  - {name}")


In [ ]:
# ── Training semua baseline dengan MLflow ────────────────────
baseline_results = {}

for model_name, model in BASELINE_MODELS.items():
    print(f"\nTraining {model_name}...")

    with mlflow.start_run(run_name=f"baseline_{model_name}"):
        # Log parameter model
        mlflow.log_params(model.get_params())
        mlflow.log_param("vectorizer_max_features", vectorizer.max_features)
        mlflow.log_param("vectorizer_ngram_range",  str(vectorizer.ngram_range))
        mlflow.log_param("train_size", len(X_train_raw))
        mlflow.log_param("test_size",  len(X_test_raw))

        # Train & evaluasi
        metrics, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test)

        # Log semua metrics
        mlflow.log_metrics(metrics)

        # Log model ke MLflow
        mlflow.sklearn.log_model(model, artifact_path="model")

        # Simpan run_id
        run_id = mlflow.active_run().info.run_id
        baseline_results[model_name] = {
            **metrics,
            "model"  : model,
            "y_pred" : y_pred,
            "run_id" : run_id,
        }

        print(f"  Accuracy   : {metrics['accuracy']:.4f}")
        print(f"  F1 Macro   : {metrics['f1_macro']:.4f}")
        print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}")
        print(f"  Run ID     : {run_id[:8]}...")

print("\n✅ Semua baseline selesai ditraining dan dicatat di MLflow")
print(f"   Buka http://127.0.0.1:5000 untuk lihat hasil")


In [ ]:
# ── Visualisasi perbandingan baseline ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(baseline_results.keys())
metrics_to_plot = {
    'F1 Macro'   : [baseline_results[m]['f1_macro']    for m in model_names],
    'F1 Weighted': [baseline_results[m]['f1_weighted']  for m in model_names],
    'Accuracy'   : [baseline_results[m]['accuracy']     for m in model_names],
}

x   = np.arange(len(model_names))
w   = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, (metric, values) in enumerate(metrics_to_plot.items()):
    axes[0].bar(x + i*w, values, w, label=metric, color=colors[i], alpha=0.85)
    for j, v in enumerate(values):
        axes[0].text(x[j] + i*w, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)

axes[0].set_title('Perbandingan Baseline Model', fontweight='bold')
axes[0].set_xticks(x + w); axes[0].set_xticklabels(model_names, rotation=10)
axes[0].set_ylabel('Score'); axes[0].set_ylim(0, 1.15)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Ranking tabel
ranking = sorted(baseline_results.items(), key=lambda x: x[1]['f1_macro'], reverse=True)
axes[1].axis('off')
table_data = [['Rank', 'Model', 'F1 Macro', 'Accuracy']]
for i, (name, res) in enumerate(ranking):
    table_data.append([
        f"#{i+1}", name,
        f"{res['f1_macro']:.4f}",
        f"{res['accuracy']:.4f}",
    ])
tbl = axes[1].table(cellText=table_data[1:], colLabels=table_data[0],
                     loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2)
for j in range(len(table_data[0])):
    tbl[(0, j)].set_facecolor('#2E4057')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')
for j in range(len(table_data[0])):
    tbl[(1, j)].set_facecolor('#ffeeba')
axes[1].set_title('Ranking Baseline Model', fontweight='bold')

plt.tight_layout()
plt.savefig(DOCS_DIR / '09_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Tentukan model terbaik
best_model_name = ranking[0][0]
print(f"\n🏆 Model terbaik: {best_model_name} (F1 Macro: {ranking[0][1]['f1_macro']:.4f})")
print(f"   → Model ini akan di-tuning dengan Optuna")


## 5. Confusion Matrix — Semua Baseline

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ['Rendah', 'Sedang', 'Tinggi']

for ax, (model_name, res) in zip(axes, baseline_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'], labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{model_name}\nF1 Macro: {res["f1_macro"]:.4f}', fontweight='bold')

plt.suptitle('Confusion Matrix — Baseline Models (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_DIR / '10_confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix disimpan")


## 6. Hyperparameter Tuning dengan Optuna

Model terbaik dari baseline akan di-tuning menggunakan Optuna
(Bayesian optimization) untuk mencari hyperparameter optimal.


In [ ]:
from sklearn.model_selection import cross_val_score

print(f"Model yang di-tuning: {best_model_name}")
print("Mulai Optuna search (estimasi 5-10 menit)...\n")

def objective(trial):
    """Fungsi objektif untuk Optuna"""

    if best_model_name == "LogisticRegression":
        params = {
            "C"           : trial.suggest_float("C", 0.01, 100, log=True),
            "solver"      : trial.suggest_categorical("solver", ["lbfgs", "saga"]),
            "max_iter"    : trial.suggest_int("max_iter", 500, 2000),
            "multi_class" : "multinomial",
            "random_state": 42,
        }
        model = LogisticRegression(**params)

    elif best_model_name == "RandomForest":
        params = {
            "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
            "max_depth"       : trial.suggest_int("max_depth", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "random_state"    : 42,
            "n_jobs"          : -1,
        }
        model = RandomForestClassifier(**params)

    else:  # LinearSVC
        params = {
            "C"       : trial.suggest_float("C", 0.01, 100, log=True),
            "max_iter": trial.suggest_int("max_iter", 1000, 3000),
        }
        model = LinearSVC(**params)

    # 5-fold cross validation pada train set
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train,
                             cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

# Jalankan Optuna
study = optuna.create_study(direction="maximize", study_name=f"tune_{best_model_name}")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✅ Optuna selesai!")
print(f"   Best F1 Macro (CV) : {study.best_value:.4f}")
print(f"   Best params        : {study.best_params}")


In [ ]:
# ── Training model tuned dengan MLflow ───────────────────────
best_params = study.best_params

if best_model_name == "LogisticRegression":
    best_params["multi_class"] = "multinomial"
    best_params["random_state"] = 42
    tuned_model = LogisticRegression(**best_params)
elif best_model_name == "RandomForest":
    best_params["random_state"] = 42
    best_params["n_jobs"] = -1
    tuned_model = RandomForestClassifier(**best_params)
else:
    tuned_model = LinearSVC(**best_params)

with mlflow.start_run(run_name=f"tuned_{best_model_name}_optuna"):
    # Log params
    mlflow.log_params(best_params)
    mlflow.log_param("tuning_method", "optuna")
    mlflow.log_param("n_trials", 50)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

    # Train pada full train set
    tuned_model.fit(X_train, y_train)
    y_pred_tuned = tuned_model.predict(X_test)

    # Metrics
    tuned_metrics = {
        "accuracy"        : accuracy_score(y_test, y_pred_tuned),
        "f1_macro"        : f1_score(y_test, y_pred_tuned, average='macro'),
        "f1_weighted"     : f1_score(y_test, y_pred_tuned, average='weighted'),
        "precision_macro" : precision_score(y_test, y_pred_tuned, average='macro'),
        "recall_macro"    : recall_score(y_test, y_pred_tuned, average='macro'),
        "optuna_best_cv"  : study.best_value,
    }
    mlflow.log_metrics(tuned_metrics)
    mlflow.sklearn.log_model(tuned_model, artifact_path="model")
    tuned_run_id = mlflow.active_run().info.run_id

print("\n=== HASIL MODEL TUNED ===")
for k, v in tuned_metrics.items():
    print(f"  {k:<20}: {v:.4f}")
print(f"\n  Run ID: {tuned_run_id[:8]}...")


In [ ]:
# ── Visualisasi Optuna ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial history
trial_numbers = [t.number for t in study.trials]
trial_values  = [t.value if t.value else 0 for t in study.trials]
best_so_far   = [max(trial_values[:i+1]) for i in range(len(trial_values))]

axes[0].plot(trial_numbers, trial_values, alpha=0.4, color='#3498db', label='Trial F1')
axes[0].plot(trial_numbers, best_so_far,  color='#e74c3c', linewidth=2, label='Best so far')
axes[0].set_title('Optuna — Trial History', fontweight='bold')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('F1 Macro (CV)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Perbandingan baseline vs tuned
all_model_names = list(baseline_results.keys()) + [f"{best_model_name} (Tuned)"]
all_f1_macro    = [baseline_results[m]['f1_macro'] for m in baseline_results] + [tuned_metrics['f1_macro']]
colors_bar      = ['#95a5a6'] * len(baseline_results) + ['#e74c3c']

bars = axes[1].bar(all_model_names, all_f1_macro, color=colors_bar, edgecolor='white', linewidth=1.5)
axes[1].set_title('Baseline vs Tuned Model', fontweight='bold')
axes[1].set_ylabel('F1 Macro (Test Set)')
axes[1].set_ylim(0, 1.1)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)

for bar, val in zip(bars, all_f1_macro):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(DOCS_DIR / '11_optuna_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot Optuna disimpan")


## 7. Evaluasi Final — Model Terbaik

In [ ]:
# ── Classification Report ─────────────────────────────────────
print("="*60)
print(f"  EVALUASI FINAL: {best_model_name} (Tuned)")
print("="*60)
print()
print(classification_report(y_test, y_pred_tuned,
                             target_names=['Rendah', 'Sedang', 'Tinggi']))


In [ ]:
# ── Confusion Matrix Final ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Rendah', 'Sedang', 'Tinggi']

cm = confusion_matrix(y_test, y_pred_tuned, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=axes[0], cmap='Blues', colorbar=True)
axes[0].set_title(f'Confusion Matrix — {best_model_name} (Tuned)', fontweight='bold')

# Normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp_norm = ConfusionMatrixDisplay(confusion_matrix=np.round(cm_norm, 2), display_labels=labels)
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=True)
axes[1].set_title(f'Confusion Matrix Normalized', fontweight='bold')

plt.tight_layout()
plt.savefig(DOCS_DIR / '12_confusion_matrix_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix final disimpan")


In [ ]:
# ── Ringkasan semua eksperimen ────────────────────────────────
print("\n=== RINGKASAN SEMUA EKSPERIMEN ===")
print(f"{'Model':<35} {'F1 Macro':>10} {'Accuracy':>10} {'F1 Weighted':>12}")
print("-" * 70)

for name, res in baseline_results.items():
    marker = " 🏆" if name == best_model_name else ""
    print(f"{'[Baseline] ' + name:<35} {res['f1_macro']:>10.4f} {res['accuracy']:>10.4f} {res['f1_weighted']:>12.4f}{marker}")

print(f"{'[Tuned] ' + best_model_name:<35} {tuned_metrics['f1_macro']:>10.4f} {tuned_metrics['accuracy']:>10.4f} {tuned_metrics['f1_weighted']:>12.4f} ✅ FINAL")
print()
improvement = tuned_metrics['f1_macro'] - baseline_results[best_model_name]['f1_macro']
print(f"Improvement setelah tuning: +{improvement:.4f} F1 Macro")
print(f"\nCek semua run di: http://127.0.0.1:5000")


## 8. Simpan Model Final

In [ ]:
# ── Simpan model dan vectorizer ───────────────────────────────
model_path      = MODEL_DIR / "model_final.pkl"
vectorizer_path = MODEL_DIR / "vectorizer.pkl"

joblib.dump(tuned_model,  model_path)
joblib.dump(vectorizer,   vectorizer_path)

print(f"✅ Model disimpan       : {model_path}")
print(f"✅ Vectorizer disimpan  : {vectorizer_path}")
print()

# ── Simpan metadata model ──────────────────────────────────────
import json
metadata = {
    "model_name"    : best_model_name,
    "model_type"    : type(tuned_model).__name__,
    "mlflow_run_id" : tuned_run_id,
    "params"        : best_params,
    "metrics"       : tuned_metrics,
    "vectorizer"    : {
        "max_features" : vectorizer.max_features,
        "ngram_range"  : list(vectorizer.ngram_range),
        "vocabulary_size": len(vectorizer.vocabulary_),
    },
    "labels"        : ['Rendah', 'Sedang', 'Tinggi'],
    "train_size"    : len(X_train_raw),
    "test_size"     : len(X_test_raw),
}

with open(MODEL_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata disimpan    :", MODEL_DIR / "model_metadata.json")
print()
print("File yang perlu di-copy ke backend:")
print(f"  backend/ml/models/model_final.pkl")
print(f"  backend/ml/models/vectorizer.pkl")
print(f"  backend/ml/models/model_metadata.json")


## 9. Quick Test — Simulasi Prediksi Real-time

In [ ]:
# ── Simulasi prediksi laporan baru ───────────────────────────
# Ini yang akan dilakukan FastAPI saat laporan masuk

def predict_urgensi(teks_laporan: str) -> dict:
    """
    Simulasi endpoint prediksi FastAPI.
    Input : teks laporan mentah dari masyarakat
    Output: label urgensi + confidence score
    """
    import sys
    sys.path.append(str(ROOT / "backend"))

    try:
        from app.ml.preprocessor import preprocess
        teks_bersih = preprocess(teks_laporan)
    except ImportError:
        # Fallback jika preprocessor belum ada di path
        teks_bersih = teks_laporan.lower()

    X = vectorizer.transform([teks_bersih])

    label = tuned_model.predict(X)[0]

    # Confidence score (probability jika tersedia)
    if hasattr(tuned_model, 'predict_proba'):
        proba = tuned_model.predict_proba(X)[0]
        confidence = float(max(proba))
    else:
        confidence = 0.0  # LinearSVC tidak punya predict_proba

    return {
        "label_urgensi"  : label,
        "confidence_score": confidence,
        "teks_bersih"    : teks_bersih[:100],
    }

# Test cases
test_cases = [
    "Seorang warga ditemukan tewas dengan luka tusukan di kawasan Surabaya Barat",
    "Pelaku narkoba berhasil diringkus polisi dengan barang bukti sabu 2 kg",
    "Terjadi vandalisme di tembok sekolah, pelaku mencoret dengan cat semprot",
]

print("=== QUICK TEST PREDIKSI ===\n")
badge = {'Tinggi': '🔴', 'Sedang': '🟡', 'Rendah': '🟢'}

for teks in test_cases:
    hasil = predict_urgensi(teks)
    print(f"Input    : {teks[:70]}...")
    print(f"Prediksi : {badge.get(hasil['label_urgensi'], '⚪')} {hasil['label_urgensi']}", end="")
    if hasil['confidence_score'] > 0:
        print(f" (confidence: {hasil['confidence_score']:.2%})")
    else:
        print()
    print()

print("✅ Model siap diintegrasikan ke FastAPI")
